<h1 style = "color : #0EE071; text-align : center;"><em>Nata Project</em> - Feature Management Notebook</h1>
<p style = "font-size : 16px; text-align: center;">This notebook has the funcion of preparing the features for the learning model.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Machine Learning I</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes, Gustavo Franco & Lucas Casimiro</p>
<br>


## <a class="anchor" id="0th-bullet">Table of Contents</a>


* [<b>1. Preparation</b>](#1st-bullet)<br>
    * [1.1 Import the needed libraries and datasets](#2nd-bullet)<br>
    * [1.2 Scaling](#4.5th-bullet)<br>
    
    
* [<b>2. Feature Selection</b>](#5th-bullet)<br>
    * [2.1 Handling Multicolinearity](#6th-bullet)<br>
    * [2.2 Filter methods](#6th-bullet)<br>
        * 2.2.1 Constant features<br>
        * 2.2.2 Spearman Correlation<br>
        * 2.2.3 Chi-Square<br>
    * [2.3 Wrapper Methods](#10th-bullet)<br>
        * 2.2.1 RFE<br>
    * [2.4 Embedded Methods](#12th-bullet)<br>
        * 2.3.1 Lasso<br>
    * [2.5 Final Insights](#15th-bullet)<br>


<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;"> 1. Preparation</h2>
<h3 style="color: #0EE071;">1.1 Import the needed libraries and datasets</h3>


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

import warnings
warnings.filterwarnings("ignore")

In [13]:
X_train = pd.read_pickle('Nata_files/data_clean/X_train_clean.pkl')
X_val = pd.read_pickle('Nata_files/data_clean/X_val_clean.pkl')
y_train = pd.read_pickle('Nata_files/data_clean/y_train_clean.pkl')
y_val = pd.read_pickle('Nata_files/data_clean/y_val_clean.pkl')

<h3 style="color: #0EE071;">1.2 Feature Scaling</h3>
<p style = "font-size : 15px;">To optimize our model, we must address the different scales and distributions of our data. In this section, we'll apply Standard Scaling to center all numerical variables at a mean of 0 and variance of 1. This prevents features with larger magnitudes from dominating the model's learning process.</p>

In [14]:
# Selecting only numerical columns for scaling
X_train_n = X_train.select_dtypes(include=['int64','float64'])
X_train_c = X_train.select_dtypes(include=['object'])
X_val_n = X_val.select_dtypes(include=['int64','float64'])
X_val_c = X_val.select_dtypes(include=['object'])

# Defining the scaler
scaler = RobustScaler()

# Fitting only the training data and transforming both 
X_train_n_scl = scaler.fit_transform(X_train_n)
X_train_n_scl = pd.DataFrame(X_train_n_scl, columns=X_train_n.columns)

X_val_n_scl = scaler.transform(X_val_n)
X_val_n_scl = pd.DataFrame(X_val_n_scl, columns=X_val_n.columns)

<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;"> 2. Feature Selection</h2>

<h3 style="color: #0EE071;">2.1 Handling Multicolinearity</h3>


As seen in the correlation heatmap in NB1, we see 'oven_temperature' and 'final_temperature' have an extremely high correlation, which means having both is redundant, so we will check for their correlation with the target variable and make a decision to drop the one with the lowest value before getting into the filter methods.

In [15]:
# Correlation analysis before dropping features
analysis_corr = pd.DataFrame(X_train_n_scl, columns=X_train_n.columns)[['oven_temperature', 'final_temperature']].copy()
analysis_corr['quality_class'] = y_train.values

corr_oven = analysis_corr['oven_temperature'].corr(analysis_corr['quality_class'])
corr_final = analysis_corr['final_temperature'].corr(analysis_corr['quality_class'])

print(f"Correlation (Oven Temp vs Target): {corr_oven}")
print(f"Correlation (Final Temp vs Target): {corr_final}")


Correlation (Oven Temp vs Target): -0.05763940542605696
Correlation (Final Temp vs Target): -0.062156860429917825


We conclude that the 'final_temperature' column has a higher absolute correlation, which means it has a better chance of helping to predict the target variable. So, we will drop 'oven_temperature'.

In [16]:
X_train.drop('oven_temperature', axis=1, inplace=True)

<h3 style="color: #0EE071;">2.2 Filter methods</h3>

### 2.2.1 Constant Features

In [17]:
X_train_n_scl.var().sort_values()

ambient_humidity     0.337547
final_temperature    0.505099
oven_temperature     0.534377
egg_temperature      0.535337
lemon_zest_ph        0.683389
salt_ratio           0.748729
vanilla_extract      0.760415
cream_fat_content    0.801421
sugar_content        0.835542
baking_duration      0.857251
preheating_time      0.961672
cooling_period       1.066615
egg_yolk_count       1.576161
dtype: float64

We conclude that all variables in the dataset exhibit non-zero variance, indicating diversity and variability in their values, so no measures need to be taken.

### 2.1.2. Spearman Correlation

We will now examine the Spearman correlation between variables.  To facilitate this analysis, a new dataframe has been created using all the training data, including the dependent variable. This inclusion allows us to investigate whether any of the independent variables exhibit correlations with the target variable. 



In [ ]:
# Joining numerical 